# Mars Crater Analysis

This Jupyter notebook contains codes for analyisng the Mars Crater Dataset, and it's a part through getting started with Data Analyis Course on Coursera.

## Importing modules and dataset.

In [2]:
##importing modules: pandas and numpy alongside the mars crater dataset
import pandas as pd
import numpy as np

data = pd.read_csv("marscrater_pds.csv", low_memory=False)

## Assigning selected columns to variables.

### Frequency distributions for our three key variables:

**DIAM_CIRCLE_IMAGE** (diameter — binned into ranges)

In [3]:
diam_freq = pd.cut(data['DIAM_CIRCLE_IMAGE'], bins=10)
diam_table = diam_freq.value_counts().sort_index().reset_index()
diam_table.columns = ['Diameter Range (km)', 'Count']
diam_table


,Diameter Range (km),Count
0,"(-0.163, 117.322]",384160
1,"(117.322, 233.644]",148
2,"(233.644, 349.966]",24
3,"(349.966, 466.288]",6
4,"(466.288, 582.61]",2
5,"(582.61, 698.932]",1
6,"(698.932, 815.254]",0
7,"(815.254, 931.576]",0
8,"(931.576, 1047.898]",0
9,"(1047.898, 1164.22]",2


**DEPTH_RIMFLOOR_TOPOG** (depth — binned)

In [4]:
depth_freq = pd.cut(data['DEPTH_RIMFLOOR_TOPOG'], bins=10)
depth_table = depth_freq.value_counts().sort_index().reset_index()
depth_table.columns = ['Depth Range (km)', 'Count']
depth_table


,Depth Range (km),Count
0,"(-0.425, 0.117]",325454
1,"(0.117, 0.654]",45769
2,"(0.654, 1.191]",10194
3,"(1.191, 1.728]",2191
4,"(1.728, 2.265]",599
5,"(2.265, 2.802]",108
6,"(2.802, 3.339]",20
7,"(3.339, 3.876]",4
8,"(3.876, 4.413]",1
9,"(4.413, 4.95]",3


**MORPHOLOGY_EJECTA_1** (ejecta type — categorical counts)

In [5]:
morph_table = data['MORPHOLOGY_EJECTA_1'].value_counts(dropna=False).reset_index()
morph_table.columns = ['Ejecta Morphology', 'Count']
morph_table

diam_table['Percentage (%)'] = (diam_table['Count'] / diam_table['Count'].sum() * 100).round(2)
diam_table


,Diameter Range (km),Count,Percentage (%)
0,"(-0.163, 117.322]",384160,99.95
1,"(117.322, 233.644]",148,0.04
2,"(233.644, 349.966]",24,0.01
3,"(349.966, 466.288]",6,0.00
4,"(466.288, 582.61]",2,0.00
5,"(582.61, 698.932]",1,0.00
6,"(698.932, 815.254]",0,0.00
7,"(815.254, 931.576]",0,0.00
8,"(931.576, 1047.898]",0,0.00
9,"(1047.898, 1164.22]",2,0.00


## What we did & what the output tells us
### What we did:

We ran a **frequency distribution**, a count of how often values appear across three variables. For continous numeric variables (diameter & depth), we grouped vaules into **10 equal width bins**. For the categorical variables (morphology), we counted each unique text label directly.

**Conclusion**: Also **heavily right-skewed**. Most craters are very shallow, many with depth = 0.0, meaning depth was **never measured** (missing data). Deep craters are rare. This is important: depth data is **incomplete** and that will affect any analysis we do relating diameter to depth.

**MORPHOLOGY_EJECTA_1 (Ejecta Type)**
The output shows very small proportions (e.g., 0.000260 = 0.026%) for the visible categories, and 156 unique morphology types exist.

→ **Conclusion:** Most craters have no recorded morphology (blank/NaN). Among those that do, ejecta types are spread across many rare categories. This means the categorical variable is sparse and dominated by missing values, which limits how much we can generalize about ejecta patterns.

## Overall Conclusions to Draw
**Small craters dominate Mars**: the size distribution is log-normal/power-law shaped, not uniform.

**Depth data is largely missing**: working with depth requires filtering out unmeasured craters (depth > 0).

**Ejecta morphology is rarely recorded**: it can only be meaningfully analyzed on a subset of the dataset.

**All three variables are non-normally distributed**: statistical methods that assume normality (like basic regression) should be applied with caution.

These insights should directly inform our next step: **choosing appropriate statistical tests and deciding whether to subset the data before deeper analysis.** 

# Data Management Decision
In this section is where we dive deeper into the analysis but before we do so, some data are missing due to various reason and one of them is because they were not recorded. Variables like **MORPHOLOGY_EJECTA_1** and **DEPTH_RIMFLOOR_TOPOG** consist data which are missing so I decided to code them out.

In [ ]:
# Create clean_data: filter depth > 0 AND morphology not empty

clean_data = data[
    (data['DEPTH_RIMFLOOR_TOPOG'] > 0) &
    (data['MORPHOLOGY_EJECTA_1'].notna ()) &
    (data['MORPHOLOGY_EJECTA_1'] != '')
].copy()

Furthermore, I will recode diameter → **diameter_category** with 4 bins, meaningful labes

In [9]:
# define the bin edges and meaningful labels
clean_data['diameter_category'] = pd.cut(clean_data['DIAM_CIRCLE_IMAGE'], bins=[0,5,50,150,1200], 
#breakpoints in km
labels= ['Small', 'Medium', 'Large', 'Basin']# Labels for each range
)

Then, I will also proceed with recoding morphology → **morphology_group** with prefix-based grouping

The Problem is our raw data has 156 unique morphology labels like:
Rd/MLERS/HuBL
SLEPd
DLEPd/Rd
SLERS
HuBL
...
and that's 
That's too fragmented to analyze meaningfully. I need to **simplify** them into groups.
*The Key Insight*: The dataset uses a prefix system. The first 2 characters tell you the fundamental ejecta type, so instead of 156 categories, we extract just the first 2 characters.

In [10]:
clean_data['morphology_group'] = clean_data['MORPHOLOGY_EJECTA_1'].str[:2]

#Then we map those prefixes to readable labels
morph_map = {
    'SL': 'Single Layer',
    'DL': 'Double Layer',
    'ML': 'Multi Layer',
    'Rd': 'Radia',
    'Hu': 'Huge',
}
clean_data['morphology_group'] = clean_data['morphology_group'].map(morph_map).fillna('Other')

And lastly, I will Create **depth_diam_ratio** secondary variable and run frequency distributions on all managed variables

In [ ]:
# creating depth_diam_ratio
clean_data['depth_diam_ratio'] = (
    clean_data['DEPTH_RIMFLOOR_TOPOG'] / clean_data['DIAM_CIRCLE_IMAGE']
)

#runing Frequency Distributions on all managed variables
ratio_freq = pd.cut(clean_data['depth_diam_ratio'], bins = 10)
ratio_table = ratio_freq.value_counts().sort_index().reset_index()
ratio_table.columns = ['Depth/Diameter Ratio', 'Count']
ratio_table['Percent (%)'] = (ratio_table['Count'] / ratio_table['Count'].sum()*100).round(2)
ratio_table

,Depth/Diameter Ratio,Count,Percent (%)
0,"(1.82e-05, 0.0233]",24750,32.22
1,"(0.0233, 0.0464]",18233,23.74
2,"(0.0464, 0.0694]",10533,13.71
3,"(0.0694, 0.0925]",9076,11.82
4,"(0.0925, 0.116]",7802,10.16
5,"(0.116, 0.139]",4596,5.98
6,"(0.139, 0.162]",1417,1.84
7,"(0.162, 0.185]",306,0.40
8,"(0.185, 0.208]",84,0.11
9,"(0.208, 0.231]",7,0.01


### The Final Tables and Observation summary.

Before diving into the numbers, it was important to clean the dataset. Two variables — MORPHOLOGY_EJECTA_1 and DEPTH_RIMFLOOR_TOPOG — contained a large number of entries where data was simply never recorded. Craters with a depth of zero and those with no ejecta label were coded out, leaving us with a focused sub-dataset of craters that have complete, meaningful measurements. This is our working dataset, clean_data.

In [15]:
# 1. diameter_category — shows the 4 meaningful bins with labels
dcat_table = clean_data['diameter_category'].value_counts().sort_index().reset_index()
dcat_table.columns = ['Diameter Category', 'Count']
dcat_table['Percentage (%)'] = (dcat_table['Count'] / dcat_table['Count'].sum() * 100).round(2)
print("=== Diameter Category Frequency ===")
dcat_table


=== Diameter Category Frequency ===


,Diameter Category,Count,Percentage (%)
0,Small,32934,42.88
1,Medium,41869,54.51
2,Large,1907,2.48
3,Basin,94,0.12


### Crater Size: Diameter Category
The size distribution of Martian craters is anything but uniform. An overwhelming majority of craters are **Small** (under 5 km in diameter), which reflects a well-known pattern in planetary science: small impactors are far more common than large ones. Medium-sized craters (5–50 km) make up a much smaller share, while Large craters and Basin-scale structures are extremely rare. Mars, like any rocky body, has been peppered far more by pebbles than boulders.

In [16]:
# 2. DEPTH_RIMFLOOR_TOPOG — binned, on clean data (no zeros)
depth_freq = pd.cut(clean_data['DEPTH_RIMFLOOR_TOPOG'], bins=10)
depth_table = depth_freq.value_counts().sort_index().reset_index()
depth_table.columns = ['Depth Range (km)', 'Count']
depth_table['Percentage (%)'] = (depth_table['Count'] / depth_table['Count'].sum() * 100).round(2)
print("=== Depth Distribution (missing data coded out) ===")
depth_table


=== Depth Distribution (missing data coded out) ===


,Depth Range (km),Count,Percentage (%)
0,"(0.00506, 0.504]",56653,73.76
1,"(0.504, 0.998]",15213,19.81
2,"(0.998, 1.492]",3567,4.64
3,"(1.492, 1.986]",1024,1.33
4,"(1.986, 2.48]",278,0.36
5,"(2.48, 2.974]",52,0.07
6,"(2.974, 3.468]",9,0.01
7,"(3.468, 3.962]",4,0.01
8,"(3.962, 4.456]",1,0.00
9,"(4.456, 4.95]",3,0.00


### Crater Depth: How Deep Do They Go?
Even after removing craters with no depth measurement, the distribution remains heavily skewed toward shallow values. Most craters with recorded depth fall between 0 and 0.65 km — relatively flat features when you consider their width. Deeper craters become progressively rarer. This isn't surprising: over billions of years, Martian craters have been slowly filled in by windblown sediment, volcanic deposits, and erosion. What we're seeing is a landscape shaped not just by impacts, but by time.

In [17]:
# 3. morphology_group — shows your 6 readable group labels
morph_table = clean_data['morphology_group'].value_counts().reset_index()
morph_table.columns = ['Ejecta Morphology Group', 'Count']
morph_table['Percentage (%)'] = (morph_table['Count'] / morph_table['Count'].sum() * 100).round(2)
print("=== Ejecta Morphology Group Frequency ===")
morph_table


=== Ejecta Morphology Group Frequency ===


,Ejecta Morphology Group,Count,Percentage (%)
0,Other,38171,49.70
1,Radia,22695,29.55
2,Single Layer,12745,16.59
3,Double Layer,2612,3.40
4,Multi Layer,581,0.76


### Ejecta Morphology: What Gets Ejected and How
When we simplified the 156 raw ejecta labels into six meaningful groups, **Single Layer (SL)** ejecta emerged as the most common type among craters with recorded morphology. Double Layer and Multi-Layer types are progressively less frequent. This pattern is geologically meaningful: single-layer ejecta typically forms in dry, rocky targets, while multi-layer ejecta is associated with volatile-rich subsurface material — ice or liquid water. The dominance of single-layer ejecta suggests that most of these craters formed in dry conditions, though the presence of multi-layer types hints at pockets of subsurface volatiles.

In [18]:
# 4. depth_diam_ratio — secondary variable, binned
ratio_freq = pd.cut(clean_data['depth_diam_ratio'], bins=10)
ratio_table = ratio_freq.value_counts().sort_index().reset_index()
ratio_table.columns = ['Depth/Diameter Ratio', 'Count']
ratio_table['Percentage (%)'] = (ratio_table['Count'] / ratio_table['Count'].sum() * 100).round(2)
print("=== Depth-to-Diameter Ratio Frequency ===")
ratio_table


=== Depth-to-Diameter Ratio Frequency ===


,Depth/Diameter Ratio,Count,Percentage (%)
0,"(1.82e-05, 0.0233]",24750,32.22
1,"(0.0233, 0.0464]",18233,23.74
2,"(0.0464, 0.0694]",10533,13.71
3,"(0.0694, 0.0925]",9076,11.82
4,"(0.0925, 0.116]",7802,10.16
5,"(0.116, 0.139]",4596,5.98
6,"(0.139, 0.162]",1417,1.84
7,"(0.162, 0.185]",306,0.40
8,"(0.185, 0.208]",84,0.11
9,"(0.208, 0.231]",7,0.01


### Depth-to-Diameter Ratio: A Window Into Crater Age
Our derived variable — the ratio of a crater's depth to its diameter — tells us how "bowl-shaped" versus "flat" a crater is. Fresh craters have higher ratios; degraded, ancient craters have lower ones. The frequency distribution shows that most craters cluster at very low ratios, meaning they are far wider than they are deep. Few craters retain the steep, bowl-like form of a fresh impact. This reinforces what the depth distribution already suggested: most of the craters in this dataset are old and eroded, survivors of a long and geologically active Martian history.

### Conclusion
Taken together, these four distributions paint a consistent picture: Mars is dominated by small, shallow, ancient craters. The rare large and deep ones are the outliers — and likely the most scientifically valuable. Every data management decision we made — coding out missing values, creating meaningful size categories, grouping ejecta types, and deriving the depth-to-diameter ratio — was designed to bring that picture into sharper focus.